# Phoneme Recognition Evaluation on TIMIT

**Paper:** [A Biomimetic Frontend for Differentiable Audio Processing](https://arxiv.org/abs/2409.08997)

**Repository:** [MeysamAmirsardari/Cortical_Front](https://github.com/MeysamAmirsardari/Cortical_Front)

This notebook loads a trained STRF-based phoneme recognition model and evaluates it
on the TIMIT test set. The model implements a differentiable auditory processing
pipeline (cochlear filterbank, power-law compression, lateral inhibition, leaky
integration) followed by Spectro-Temporal Receptive Field (STRF) encoding and a
CNN decoder.

**Pipeline:**
1. Raw audio (16 kHz) &rarr; Cochlear filterbank (129 channels)
2. Power-law compression &rarr; Lateral inhibition &rarr; Leaky integration
3. Downsampling &rarr; Auditory spectrogram (200 frames &times; 128 channels per second)
4. STRF encoding (40 learnable scale-rate filters)
5. CNN decoder &rarr; 62 phoneme classes + blank

**Runtime:** Select **GPU** (Runtime &rarr; Change runtime type &rarr; T4 GPU) for faster JAX execution.

## 1. Environment Setup

In [ ]:
%%capture
# Install JAX with GPU support, Flax, and other dependencies
# Colab already has torch, numpy, matplotlib pre-installed
!pip install --upgrade jax[cuda12] flax optax
!pip install librosa editdistance soundfile kagglehub

In [ ]:
# Clone the repository
import os

REPO_URL = "https://github.com/MeysamAmirsardari/Cortical_Front.git"
REPO_DIR = "/content/Cortical_Front"
CODE_DIR = os.path.join(REPO_DIR, "r_code")

if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}
    print(f"Cloned repository to {REPO_DIR}")
else:
    print(f"Repository already exists at {REPO_DIR}")

# Change to the code directory -- the model code expects this as CWD
os.chdir(CODE_DIR)
print(f"Working directory: {os.getcwd()}")

In [ ]:
import sys
import pickle
import warnings
from pathlib import Path

import numpy as np
import jax
jax.config.update("jax_enable_x64", True)
import jax.numpy as jnp
from jax import jit

import torch
from torch.utils import data as torch_data

import librosa
import editdistance
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from IPython.display import Audio, display

# Add code directory to Python path
if CODE_DIR not in sys.path:
    sys.path.insert(0, CODE_DIR)

# Import model components from the repository
from strfpy_jax import initialize_sr, initialize_compression_params, strf, wav2aud_j, read_cochba_j
from supervisedSTRF import vSupervisedSTRF

print(f"JAX version: {jax.__version__}")
print(f"JAX backend: {jax.default_backend()}")
print(f"JAX devices: {jax.devices()}")
print(f"PyTorch version: {torch.__version__}")

## 2. Load Trained Model Checkpoint

The trained checkpoints are stored as Python pickle files with the structure `[nn_params, params]`:
- **`nn_params`**: Flax parameter tree (Conv/Dense weights for the CNN decoder, LIN kernel, compression alphas, leaky integration alpha)
- **`params`**: Auditory processing parameters dict with keys:
  - `sr`: STRF scale-rate parameters, shape `(n_strfs, 2)`
  - `compression_params`: per-channel power-law exponents, shape `(129,)`
  - `alpha` *(optional)*: leaky integration time constant

We auto-detect the model configuration from the checkpoint contents.

In [ ]:
# ---------- Configuration ----------
# Path to checkpoint (relative to r_code/)
CHECKPOINT_PATH = "nishit_trained_models/main_jax_phoneme_rec_timit/chkStep_200000.p"

# These must match the training configuration.
# The default values below match the paper's phoneme recognition setup (Section 3.1).
MODEL_CONFIG = dict(
    encoder_type  = "strf",
    decoder_type  = "cnn",
    input_type    = "audio",
    conv_feats    = [10, 20, 40],
    pooling_stride = 2,
)
# ---------- End Configuration ----------

# Load checkpoint
ckpt_path = Path(CHECKPOINT_PATH)
assert ckpt_path.exists(), f"Checkpoint not found: {ckpt_path.resolve()}"

with open(ckpt_path, "rb") as f:
    checkpoint = pickle.load(f)

assert isinstance(checkpoint, (list, tuple)) and len(checkpoint) == 2, \
    f"Expected [nn_params, params], got {type(checkpoint)}"

nn_params, aud_params = checkpoint

print("Checkpoint loaded successfully!")
print(f"  Path: {ckpt_path}")
print(f"  nn_params type: {type(nn_params)}")
print(f"  aud_params keys: {list(aud_params.keys()) if hasattr(aud_params, 'keys') else 'N/A'}")

In [ ]:
# Auto-detect model configuration from checkpoint contents

from flax.traverse_util import flatten_dict
import re

def detect_model_config(nn_params, aud_params):
    """Infer model hyperparameters directly from checkpoint weights."""
    config = {}

    # --- n_phones: from the last Dense kernel output dimension ---
    params_dict = nn_params.get("params", nn_params)
    flat = flatten_dict(params_dict)
    best_idx, best_out_dim = -1, None
    for key, val in flat.items():
        key_str = "/".join(str(k) for k in key)
        if "Dense" in key_str and key[-1] == "kernel" and hasattr(val, "shape"):
            m = re.search(r"Dense_(\d+)", key_str)
            idx = int(m.group(1)) if m else 0
            if idx >= best_idx:
                best_idx = idx
                best_out_dim = int(val.shape[-1])
    # The model adds +1 internally: Dense(features=n_phones+1)
    config["n_phones"] = best_out_dim - 1 if best_out_dim else 62

    # --- compression_method: if 'compression_params' in aud_params ---
    has_compression = "compression_params" in aud_params
    config["compression_method"] = "power" if has_compression else "identity"

    # --- update_lin: if 'alpha' in aud_params ---
    config["update_lin"] = "alpha" in aud_params

    # --- num_strfs: from shape of sr ---
    config["num_strfs"] = int(aud_params["sr"].shape[0])

    # --- use_class: check if AuditorySpectrogram params exist ---
    params_str = str(list(flat.keys()))
    config["use_class"] = "AuditorySpectrogram" in params_str

    return config

detected = detect_model_config(nn_params, aud_params)
MODEL_CONFIG.update(detected)

print("Detected model configuration:")
for k, v in sorted(MODEL_CONFIG.items()):
    print(f"  {k:25s} = {v}")

In [ ]:
# Instantiate the model with detected configuration
model = vSupervisedSTRF(
    n_phones           = MODEL_CONFIG["n_phones"],
    input_type         = MODEL_CONFIG["input_type"],
    encoder_type       = MODEL_CONFIG["encoder_type"],
    decoder_type       = MODEL_CONFIG["decoder_type"],
    compression_method = MODEL_CONFIG["compression_method"],
    update_lin         = MODEL_CONFIG["update_lin"],
    use_class          = MODEL_CONFIG["use_class"],
    conv_feats         = MODEL_CONFIG["conv_feats"],
    pooling_stride     = MODEL_CONFIG["pooling_stride"],
)

# Verify checkpoint is compatible: do a dummy forward pass, then replace with loaded weights
BATCH_SIZE = 4
dummy_input = jnp.ones([BATCH_SIZE, 16000])
_ = model.init(jax.random.key(0), dummy_input, aud_params)

# JIT-compile the forward pass
@jit
def predict_batch(nn_params, audio_batch, params):
    """Run forward pass and return per-frame logits and predictions."""
    logits = model.apply(nn_params, audio_batch, params)
    # If output is 4D (batch, time, pooled, n_phones), average over pooled dim
    if logits.ndim == 4:
        logits = jnp.mean(logits, axis=2)
    predictions = jnp.argmax(logits, axis=-1)  # (batch, time)
    return logits, predictions

# Warm up JIT compilation with a dummy batch
_ = predict_batch(nn_params, dummy_input, aud_params)
print("Model initialized and JIT-compiled successfully.")

## 3. Load TIMIT Test Data

We download the TIMIT dataset from Kaggle ([DARPA TIMIT](https://www.kaggle.com/datasets/mfekadu/darpa-timit-acousticphonetic-continuous-speech)) and parse the `.PHN` phoneme alignment files directly.

Each `.PHN` file contains lines of `start_sample stop_sample phone`. We convert these to
frame-level labels at 5 ms resolution (200 frames per second), matching the training pipeline.

> **Kaggle credentials:** You need a Kaggle account. In Colab, go to the left sidebar
> **Secrets** (key icon), add `KAGGLE_USERNAME` and `KAGGLE_KEY` from your
> [kaggle.com/settings](https://www.kaggle.com/settings) API section.

In [ ]:
# TIMIT 61-phone inventory (same order as precompute_timit_labels.py)
# Labels are 1-indexed: phone index i maps to TIMIT_PHONEMES[i-1]
TIMIT_PHONEMES = [
    "aa", "ae", "ah", "ao", "aw", "ax", "ax-h", "axr", "ay",
    "b", "bcl", "ch", "d", "dcl", "dh", "dx",
    "eh", "el", "em", "en", "eng", "epi", "er", "ey",
    "f", "g", "gcl", "h#", "hh", "hv",
    "ih", "ix", "iy",
    "jh", "k", "kcl", "l", "m", "n", "ng", "nx",
    "ow", "oy", "p", "pau", "pcl", "q", "r",
    "s", "sh", "t", "tcl", "th", "uh", "uw", "ux",
    "v", "w", "y", "z", "zh",
]
PHONEME_TO_IDX = {p: i + 1 for i, p in enumerate(TIMIT_PHONEMES)}  # 1-indexed

# Standard TIMIT 61 -> 39 phone folding for evaluation
PHONE_61_TO_39 = {
    "aa": "aa", "ae": "ae", "ah": "ah", "ao": "aa", "aw": "aw",
    "ax": "ah", "ax-h": "ah", "axr": "er", "ay": "ay", "b": "b",
    "bcl": "sil", "ch": "ch", "d": "d", "dcl": "sil", "dh": "dh",
    "dx": "dx", "eh": "eh", "el": "l", "em": "m", "en": "n",
    "eng": "ng", "epi": "sil", "er": "er", "ey": "ey", "f": "f",
    "g": "g", "gcl": "sil", "h#": "sil", "hh": "hh", "hv": "hh",
    "ih": "ih", "ix": "ih", "iy": "iy", "jh": "jh", "k": "k",
    "kcl": "sil", "l": "l", "m": "m", "n": "n", "ng": "ng",
    "nx": "n", "ow": "ow", "oy": "oy", "p": "p", "pau": "sil",
    "pcl": "sil", "q": "", "r": "r", "s": "s", "sh": "sh",
    "t": "t", "tcl": "sil", "th": "th", "uh": "uh", "uw": "uw",
    "ux": "uw", "v": "v", "w": "w", "y": "y", "z": "z", "zh": "sh",
}

def idx_to_phone(idx):
    """Map 1-indexed label to phone string."""
    if idx == 0:
        return "<blank>"
    if 1 <= idx <= len(TIMIT_PHONEMES):
        return TIMIT_PHONEMES[idx - 1]
    return f"<unk:{idx}>"

print(f"Phone inventory: {len(TIMIT_PHONEMES)} phones (61)")
print(f"39-phone folded set: {len(set(PHONE_61_TO_39.values()) - {''})} phones")

In [ ]:
import kagglehub
from pathlib import Path
import glob as glob_module

# Download TIMIT from Kaggle (cached after first download)
kaggle_path = kagglehub.dataset_download("mfekadu/darpa-timit-acousticphonetic-continuous-speech")
print(f"Dataset downloaded to: {kaggle_path}")

# Locate the TIMIT root (handles varying nesting levels)
kaggle_path = Path(kaggle_path)
candidates = list(kaggle_path.rglob("TEST"))
assert len(candidates) > 0, (
    f"Could not find TEST/ directory under {kaggle_path}. "
    "Check the Kaggle download structure."
)
TIMIT_ROOT = candidates[0].parent  # parent of TEST/
print(f"TIMIT root: {TIMIT_ROOT}")

# Find all .PHN files in TEST split (case-insensitive)
phn_files = sorted(
    list(TIMIT_ROOT.rglob("TEST/**/*.[Pp][Hh][Nn]"))
)
print(f"Found {len(phn_files)} .PHN files in TEST split")

# Show a few paths for sanity check
for p in phn_files[:3]:
    print(f"  {p.relative_to(TIMIT_ROOT)}")

In [ ]:
SR = 16000          # Sample rate
FRAME_MS = 5        # Frame length in ms (matches training)
FRAME_SAMPLES = int(SR * FRAME_MS / 1000)   # 80 samples per frame
N_SAMPLES = SR      # 1 second = 16000 samples
N_FRAMES = 200      # Frames per 1-second segment
MIN_FRAMES = 215    # Minimum utterance length (frames) to allow random offset


def parse_phn_file(phn_path):
    """Parse a TIMIT .PHN file and return (start_sample, end_sample, phone_idx) tuples."""
    entries = []
    with open(phn_path, "r") as f:
        for line in f:
            parts = line.strip().split()
            if len(parts) >= 3:
                start = int(parts[0])
                end = int(parts[1])
                phone = parts[2].lower()
                idx = PHONEME_TO_IDX.get(phone, 1)  # default to 1 if unknown
                entries.append((start, end, idx))
    return entries


def make_frame_labels_from_phn(phn_entries, total_samples):
    """Convert parsed PHN entries to dense frame-level labels.

    Returns an array of shape (n_total_frames,) with 1-indexed phone labels.
    """
    n_frames = total_samples // FRAME_SAMPLES
    labels = np.ones(n_frames, dtype=np.int64)  # default silence
    for start, end, idx in phn_entries:
        sf = start // FRAME_SAMPLES
        ef = end // FRAME_SAMPLES
        labels[sf : min(ef, n_frames)] = idx
    return labels


def find_wav_for_phn(phn_path):
    """Find the corresponding WAV file for a .PHN file (handles case variations)."""
    stem = phn_path.stem
    parent = phn_path.parent
    for ext in [".WAV", ".wav", ".Wav"]:
        wav_path = parent / (stem + ext)
        if wav_path.exists():
            return wav_path
    return None


class TIMITTestDataset(torch_data.Dataset):
    """Loads TIMIT test utterances from Kaggle-downloaded files."""

    def __init__(self, phn_files, max_samples=None):
        self.items = []
        for phn_path in phn_files:
            if max_samples and len(self.items) >= max_samples:
                break

            wav_path = find_wav_for_phn(phn_path)
            if wav_path is None:
                continue

            # Load audio (librosa handles NIST SPHERE headers)
            try:
                audio, _ = librosa.load(str(wav_path), sr=SR, mono=True)
            except Exception:
                continue
            audio = audio.astype(np.float32)

            n_frames_total = len(audio) // FRAME_SAMPLES
            if n_frames_total < MIN_FRAMES:
                continue  # too short for a 1-second segment

            phn_entries = parse_phn_file(phn_path)
            labels = make_frame_labels_from_phn(phn_entries, len(audio))
            self.items.append((audio, labels))

    def __len__(self):
        return len(self.items)

    def __getitem__(self, idx):
        audio, labels = self.items[idx]
        n_frames_total = len(labels)

        # Pick a random 1-second segment (same logic as training)
        max_start = n_frames_total - MIN_FRAMES
        frame_start = np.random.randint(0, max(max_start, 1))
        seg_labels = labels[frame_start : frame_start + N_FRAMES]
        if len(seg_labels) < N_FRAMES:
            seg_labels = np.pad(seg_labels, (0, N_FRAMES - len(seg_labels)),
                                mode="constant", constant_values=1)

        # Extract corresponding audio
        sample_start = frame_start * FRAME_SAMPLES
        seg_audio = audio[sample_start : sample_start + N_SAMPLES]
        if len(seg_audio) < N_SAMPLES:
            seg_audio = np.pad(seg_audio, (0, N_SAMPLES - len(seg_audio)))

        # RMS normalization (matches training)
        rms = np.sqrt(np.mean(seg_audio ** 2))
        if rms > 0:
            seg_audio = seg_audio / rms

        return seg_audio, seg_labels


# ---------- Build dataset ----------
# Use a subset for faster evaluation; set to None for the full test set
NUM_TEST_SAMPLES = 200  # Evaluate on 200 utterances (set to None for all ~1680)

test_dataset = TIMITTestDataset(phn_files, max_samples=NUM_TEST_SAMPLES)
print(f"Test dataset: {len(test_dataset)} segments (from {NUM_TEST_SAMPLES or 'all'} utterances)")

## 4. Run Inference

We run the model on the test set in mini-batches, collecting per-frame predictions and ground-truth labels.

In [ ]:
from tqdm.notebook import tqdm

test_loader = torch_data.DataLoader(
    test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=0, drop_last=False
)

all_preds = []       # list of (200,) arrays -- raw frame predictions
all_labels = []      # list of (200,) arrays -- ground truth frame labels
all_logits_list = [] # keep first few for visualization

for batch_idx, (audio_batch, label_batch) in enumerate(tqdm(test_loader, desc="Inference")):
    audio_np = audio_batch.numpy()
    labels_np = label_batch.numpy()
    bs = audio_np.shape[0]

    # Pad last batch if smaller than BATCH_SIZE (model expects fixed batch dim)
    if bs < BATCH_SIZE:
        pad = np.zeros((BATCH_SIZE - bs, N_SAMPLES), dtype=np.float32)
        audio_np = np.concatenate([audio_np, pad], axis=0)

    x = jnp.array(audio_np)
    logits, preds = predict_batch(nn_params, x, aud_params)

    preds_np = np.array(preds)[:bs]
    logits_np = np.array(logits)[:bs]

    for i in range(bs):
        all_preds.append(preds_np[i])
        all_labels.append(labels_np[i])
        if len(all_logits_list) < 20:
            all_logits_list.append(logits_np[i])

print(f"\nInference complete: {len(all_preds)} segments processed.")

## 5. Evaluation: Phoneme Error Rate (PER)

We compute PER using edit distance between collapsed prediction sequences and ground truth.
Following TIMIT convention, we report PER with 61&rarr;39 phone folding.

In [ ]:
def collapse_repeated(seq):
    """Collapse consecutive identical labels: [1,1,2,2,3] -> [1,2,3]."""
    if len(seq) == 0:
        return []
    out = [seq[0]]
    for x in seq[1:]:
        if x != out[-1]:
            out.append(x)
    return out


def fold_61_to_39(seq_61_idx):
    """Convert a sequence of 1-indexed 61-phone labels to 39-phone indices."""
    phone_39_set = sorted(set(PHONE_61_TO_39.values()) - {""})
    phone_39_to_idx = {p: i for i, p in enumerate(phone_39_set)}
    seq_39 = []
    for idx in seq_61_idx:
        if idx < 1 or idx > len(TIMIT_PHONEMES):
            continue
        phone_61 = TIMIT_PHONEMES[idx - 1]
        phone_39 = PHONE_61_TO_39.get(phone_61, "")
        if phone_39 and phone_39 in phone_39_to_idx:
            seq_39.append(phone_39_to_idx[phone_39])
    return seq_39


def compute_per(all_preds, all_labels, use_39_fold=True):
    """Compute Phoneme Error Rate over all segments."""
    total_dist = 0
    total_len = 0

    for pred, label in zip(all_preds, all_labels):
        # Collapse repeated frames to get phone sequences
        pred_seq = collapse_repeated(pred.tolist())
        label_seq = collapse_repeated(label.tolist())

        # Remove blank predictions (index 0)
        pred_seq = [p for p in pred_seq if p != 0]
        label_seq = [l for l in label_seq if l != 0]

        if use_39_fold:
            pred_seq = fold_61_to_39(pred_seq)
            label_seq = fold_61_to_39(label_seq)

        dist = editdistance.eval(pred_seq, label_seq)
        total_dist += dist
        total_len += len(label_seq)

    per = (total_dist / total_len * 100) if total_len > 0 else 0.0
    return per, total_dist, total_len


# --- Frame-level accuracy (before collapsing) ---
all_preds_flat = np.concatenate(all_preds)
all_labels_flat = np.concatenate(all_labels)
# Exclude blank (0) frames from accuracy
mask = all_labels_flat > 0
frame_acc = np.mean(all_preds_flat[mask] == all_labels_flat[mask]) * 100

# --- PER with 39-phone folding ---
per_39, dist_39, len_39 = compute_per(all_preds, all_labels, use_39_fold=True)

# --- PER with 61 phones ---
per_61, dist_61, len_61 = compute_per(all_preds, all_labels, use_39_fold=False)

print("=" * 60)
print("EVALUATION RESULTS")
print("=" * 60)
print(f"  Segments evaluated:       {len(all_preds)}")
print(f"  Frame-level accuracy:     {frame_acc:.2f}%")
print(f"  PER (39 phones, folded):  {per_39:.2f}%  (edit dist {dist_39} / {len_39} phones)")
print(f"  PER (61 phones):          {per_61:.2f}%  (edit dist {dist_61} / {len_61} phones)")
print("=" * 60)

## 6. Visualizations

### 6.1 Sample Predictions vs Ground Truth

For a few test segments, we plot the waveform, auditory spectrogram, and predicted vs ground-truth phoneme labels aligned in time.

In [ ]:
def plot_sample(sample_idx, all_preds, all_labels, all_logits_list, test_dataset):
    """Plot waveform, softmax posteriors, and predicted vs GT phone labels for one segment."""
    pred = all_preds[sample_idx]
    label = all_labels[sample_idx]
    logits = all_logits_list[sample_idx]

    # Get audio for this sample
    audio, _ = test_dataset[sample_idx]
    time_axis = np.arange(N_FRAMES) * FRAME_MS / 1000  # seconds

    # Softmax posteriors
    posteriors = jax.nn.softmax(jnp.array(logits), axis=-1)
    posteriors = np.array(posteriors)

    fig = plt.figure(figsize=(16, 8))
    gs = gridspec.GridSpec(4, 1, height_ratios=[1, 2, 0.6, 0.6], hspace=0.35)

    # --- Waveform ---
    ax0 = fig.add_subplot(gs[0])
    t_audio = np.arange(len(audio)) / SR
    ax0.plot(t_audio, audio, linewidth=0.3, color="steelblue")
    ax0.set_xlim(0, 1)
    ax0.set_ylabel("Amplitude")
    ax0.set_title(f"Sample {sample_idx}", fontsize=13, fontweight="bold")
    ax0.tick_params(labelbottom=False)

    # --- Softmax posteriors heatmap ---
    ax1 = fig.add_subplot(gs[1])
    im = ax1.imshow(posteriors.T, aspect="auto", origin="lower",
                     extent=[0, 1, 0, posteriors.shape[1]], cmap="magma")
    ax1.set_ylabel("Phone class")
    ax1.set_xlabel("")
    ax1.tick_params(labelbottom=False)
    plt.colorbar(im, ax=ax1, fraction=0.02, pad=0.01, label="P(phone)")

    # --- Ground truth labels ---
    ax2 = fig.add_subplot(gs[2])
    ax2.imshow(label[None, :], aspect="auto", cmap="tab20",
               extent=[0, 1, 0, 1], interpolation="nearest")
    # Add phone text labels
    gt_collapsed = collapse_repeated(label.tolist())
    boundaries = []
    pos = 0
    for phone_idx in gt_collapsed:
        count = 0
        while pos < len(label) and label[pos] == phone_idx:
            count += 1
            pos += 1
        mid = (pos - count / 2) * FRAME_MS / 1000
        if count > 3:
            ax2.text(mid, 0.5, idx_to_phone(phone_idx), ha="center", va="center",
                     fontsize=6, color="white", fontweight="bold")
    ax2.set_ylabel("GT")
    ax2.set_yticks([])
    ax2.tick_params(labelbottom=False)

    # --- Predicted labels ---
    ax3 = fig.add_subplot(gs[3])
    ax3.imshow(pred[None, :], aspect="auto", cmap="tab20",
               extent=[0, 1, 0, 1], interpolation="nearest")
    pred_collapsed = collapse_repeated(pred.tolist())
    pos = 0
    for phone_idx in pred_collapsed:
        count = 0
        while pos < len(pred) and pred[pos] == phone_idx:
            count += 1
            pos += 1
        mid = (pos - count / 2) * FRAME_MS / 1000
        if count > 3:
            ax3.text(mid, 0.5, idx_to_phone(phone_idx), ha="center", va="center",
                     fontsize=6, color="white", fontweight="bold")
    ax3.set_ylabel("Pred")
    ax3.set_yticks([])
    ax3.set_xlabel("Time (s)")

    plt.tight_layout()
    plt.show()

    # Print collapsed phone sequences
    gt_phones = [idx_to_phone(i) for i in collapse_repeated(label.tolist()) if i > 0]
    pred_phones = [idx_to_phone(i) for i in collapse_repeated(pred.tolist()) if i > 0]
    print(f"  GT:   {' '.join(gt_phones)}")
    print(f"  Pred: {' '.join(pred_phones)}")
    print()


# Plot first 5 samples
for i in range(min(5, len(all_logits_list))):
    plot_sample(i, all_preds, all_labels, all_logits_list, test_dataset)

### 6.2 Confusion Matrix (39-phone folded)

Confusion matrix over all frame-level predictions, after folding 61 phones to the standard 39-phone set.

In [ ]:
from collections import Counter

# Build 39-phone mapping for frame-level confusion
phone_39_set = sorted(set(PHONE_61_TO_39.values()) - {""})
phone_39_to_idx = {p: i for i, p in enumerate(phone_39_set)}
n_39 = len(phone_39_set)


def idx61_to_idx39(idx_61):
    """Map a single 1-indexed 61-phone label to 0-indexed 39-phone label, or -1."""
    if idx_61 < 1 or idx_61 > len(TIMIT_PHONEMES):
        return -1
    phone_61 = TIMIT_PHONEMES[idx_61 - 1]
    phone_39 = PHONE_61_TO_39.get(phone_61, "")
    if phone_39 and phone_39 in phone_39_to_idx:
        return phone_39_to_idx[phone_39]
    return -1


# Build confusion matrix
confusion = np.zeros((n_39, n_39), dtype=np.int64)
for pred, label in zip(all_preds, all_labels):
    for p, l in zip(pred, label):
        p39 = idx61_to_idx39(int(p))
        l39 = idx61_to_idx39(int(l))
        if p39 >= 0 and l39 >= 0:
            confusion[l39, p39] += 1

# Normalize rows to percentages
row_sums = confusion.sum(axis=1, keepdims=True)
row_sums[row_sums == 0] = 1
confusion_norm = confusion / row_sums * 100

fig, ax = plt.subplots(figsize=(14, 12))
im = ax.imshow(confusion_norm, cmap="Blues", aspect="auto", vmin=0, vmax=100)
ax.set_xticks(range(n_39))
ax.set_yticks(range(n_39))
ax.set_xticklabels(phone_39_set, rotation=90, fontsize=7)
ax.set_yticklabels(phone_39_set, fontsize=7)
ax.set_xlabel("Predicted phone", fontsize=11)
ax.set_ylabel("True phone", fontsize=11)
ax.set_title("Frame-level Confusion Matrix (39 phones, row-normalized %)", fontsize=13)
plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04, label="% of true class")
plt.tight_layout()
plt.show()

# Print top confusions
print("\nTop 10 most confused phone pairs (true -> pred, count):")
off_diag = []
for i in range(n_39):
    for j in range(n_39):
        if i != j and confusion[i, j] > 0:
            off_diag.append((confusion[i, j], phone_39_set[i], phone_39_set[j]))
off_diag.sort(reverse=True)
for count, true_p, pred_p in off_diag[:10]:
    print(f"  {true_p:>5s} -> {pred_p:<5s}  ({count} frames)")

### 6.3 Learned STRF Parameters

Visualize the learned scale-rate parameters of the 40 STRFs. Each STRF is characterized by a (scale, rate) pair that determines its spectral and temporal modulation selectivity.

In [ ]:
sr = np.array(aud_params["sr"])
scales = sr[:, 0]
rates = sr[:, 1]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Scatter plot of (scale, rate) pairs
ax = axes[0]
scatter = ax.scatter(rates, scales, c=np.arange(len(sr)), cmap="viridis",
                     s=80, edgecolors="black", linewidth=0.5, zorder=3)
ax.axhline(0, color="gray", linewidth=0.5, linestyle="--")
ax.axvline(0, color="gray", linewidth=0.5, linestyle="--")
ax.set_xlabel("Rate (Hz) - temporal modulation", fontsize=11)
ax.set_ylabel("Scale (cyc/oct) - spectral modulation", fontsize=11)
ax.set_title("Learned STRF Scale-Rate Parameters", fontsize=13)
ax.grid(True, alpha=0.3)
plt.colorbar(scatter, ax=ax, label="STRF index")

# Histograms
ax = axes[1]
ax.hist(np.abs(scales), bins=15, alpha=0.6, label="|Scale|", color="steelblue")
ax.hist(np.abs(rates), bins=15, alpha=0.6, label="|Rate|", color="coral")
ax.set_xlabel("Parameter magnitude", fontsize=11)
ax.set_ylabel("Count", fontsize=11)
ax.set_title("Distribution of |Scale| and |Rate|", fontsize=13)
ax.legend()

plt.tight_layout()
plt.show()

print(f"Scale range: [{scales.min():.3f}, {scales.max():.3f}]")
print(f"Rate range:  [{rates.min():.3f}, {rates.max():.3f}]")

### 6.4 Auditory Spectrogram Examples

Visualize the biomimetic auditory spectrogram produced by the differentiable frontend for a few test samples. This is the intermediate representation after the cochlear filterbank, power-law compression, lateral inhibition, and leaky integration stages.

In [ ]:
# Compute auditory spectrograms via the model's encode step
# We use the non-batched pipeline from strfpy_jax for clarity

Bs, As = read_cochba_j()

@jit
def compute_auditory_spectrogram(x, compression_params):
    """Compute the auditory spectrogram for a single 1-second waveform.

    Uses the same pipeline as wav2aud_j with power-law compression:
      cochlear filterbank -> compression -> first diff -> ReLU -> leaky integration -> downsample
    """
    return wav2aud_j(x, 5, 8, compression_params, 0, As, Bs,
                     fft=True, compression_method="power",
                     return_stage=5).T  # (200, 128)

fig, axes = plt.subplots(3, 2, figsize=(16, 10))

for i in range(min(3, len(test_dataset))):
    audio, labels = test_dataset[i]
    x = jnp.array(audio)

    # Try to compute auditory spectrogram; fall back to mel if it fails
    try:
        cp = aud_params.get("compression_params", jnp.ones(129))
        aud_spec = compute_auditory_spectrogram(x, cp)
        aud_spec = np.array(aud_spec.real)
    except Exception as e:
        print(f"  Sample {i}: auditory spectrogram failed ({e}), using mel fallback")
        aud_spec = librosa.feature.melspectrogram(
            y=audio, sr=SR, n_fft=512, hop_length=80, n_mels=128)[:, :200].T
        aud_spec = np.log1p(aud_spec)

    # Waveform
    ax = axes[i, 0]
    t = np.arange(len(audio)) / SR
    ax.plot(t, audio, linewidth=0.3, color="steelblue")
    ax.set_xlim(0, 1)
    ax.set_ylabel("Amplitude")
    if i == 0:
        ax.set_title("Waveform", fontsize=12, fontweight="bold")
    if i == 2:
        ax.set_xlabel("Time (s)")

    # Auditory spectrogram
    ax = axes[i, 1]
    im = ax.imshow(aud_spec.T, aspect="auto", origin="lower",
                    extent=[0, 1, 0, 128], cmap="inferno")
    ax.set_ylabel("Frequency channel")
    if i == 0:
        ax.set_title("Auditory Spectrogram", fontsize=12, fontweight="bold")
    if i == 2:
        ax.set_xlabel("Time (s)")
    plt.colorbar(im, ax=ax, fraction=0.02, pad=0.01)

plt.tight_layout()
plt.show()

### 6.5 Per-Phone Accuracy (39-phone set)

Bar chart showing recognition accuracy for each of the 39 phones.

In [ ]:
# Per-phone accuracy from the confusion matrix diagonal
diag = np.diag(confusion)
per_phone_acc = np.where(row_sums.flatten() > 0,
                          diag / row_sums.flatten() * 100, 0)

# Sort by accuracy
sorted_idx = np.argsort(per_phone_acc)

fig, ax = plt.subplots(figsize=(14, 6))
colors = plt.cm.RdYlGn(per_phone_acc[sorted_idx] / 100)
bars = ax.barh(range(n_39), per_phone_acc[sorted_idx], color=colors, edgecolor="gray", linewidth=0.3)
ax.set_yticks(range(n_39))
ax.set_yticklabels([phone_39_set[i] for i in sorted_idx], fontsize=8)
ax.set_xlabel("Frame-level Accuracy (%)", fontsize=11)
ax.set_title("Per-Phone Recognition Accuracy (39-phone set)", fontsize=13)
ax.axvline(50, color="gray", linewidth=0.5, linestyle="--", alpha=0.5)
ax.set_xlim(0, 105)

# Annotate bars with count
for i, idx in enumerate(sorted_idx):
    count = int(row_sums.flatten()[idx])
    ax.text(per_phone_acc[idx] + 1, i, f"{per_phone_acc[idx]:.0f}% (n={count})",
            va="center", fontsize=7)

plt.tight_layout()
plt.show()

print(f"\nOverall frame-level accuracy (39 phones): "
      f"{np.sum(diag) / np.sum(confusion) * 100:.2f}%")

### 6.6 Listen to Test Samples

Play a few test audio samples with their predicted and ground-truth phone sequences.

In [ ]:
for i in range(min(5, len(test_dataset))):
    audio, _ = test_dataset[i]
    pred = all_preds[i]
    label = all_labels[i]

    gt_phones = " ".join(idx_to_phone(j) for j in collapse_repeated(label.tolist()) if j > 0)
    pred_phones = " ".join(idx_to_phone(j) for j in collapse_repeated(pred.tolist()) if j > 0)

    print(f"--- Sample {i} ---")
    print(f"  GT:   {gt_phones}")
    print(f"  Pred: {pred_phones}")
    display(Audio(audio, rate=SR))

## 7. Checkpoint Comparison (Optional)

Compare PER across different training steps to see learning progression.

In [ ]:
# Evaluate a few checkpoints across training to see learning curve
# Set SKIP_COMPARISON = True to skip this (takes a few minutes)

SKIP_COMPARISON = False
CKPT_DIR = Path("nishit_trained_models/main_jax_phoneme_rec_timit")
STEPS_TO_EVAL = [10000, 50000, 100000, 150000, 200000]

if not SKIP_COMPARISON:
    # Use a small subset for quick comparison
    quick_dataset = TIMITTestDataset(timit, max_samples=50)
    quick_loader = torch_data.DataLoader(
        quick_dataset, batch_size=BATCH_SIZE, shuffle=False, drop_last=False
    )

    results = []
    for step in STEPS_TO_EVAL:
        ckpt_file = CKPT_DIR / f"chkStep_{step}.p"
        if not ckpt_file.exists():
            print(f"  Skipping step {step} (file not found)")
            continue

        with open(ckpt_file, "rb") as f:
            ckpt = pickle.load(f)
        nn_p, aud_p = ckpt

        step_preds, step_labels = [], []
        for audio_batch, label_batch in quick_loader:
            audio_np = audio_batch.numpy()
            labels_np = label_batch.numpy()
            bs = audio_np.shape[0]
            if bs < BATCH_SIZE:
                audio_np = np.concatenate(
                    [audio_np, np.zeros((BATCH_SIZE - bs, N_SAMPLES), dtype=np.float32)])
            _, preds = predict_batch(nn_p, jnp.array(audio_np), aud_p)
            preds_np = np.array(preds)[:bs]
            for j in range(bs):
                step_preds.append(preds_np[j])
                step_labels.append(labels_np[j])

        per, _, _ = compute_per(step_preds, step_labels, use_39_fold=True)
        results.append((step, per))
        print(f"  Step {step:>7d}: PER = {per:.2f}%")

    if results:
        steps, pers = zip(*results)
        fig, ax = plt.subplots(figsize=(10, 5))
        ax.plot(steps, pers, "o-", color="steelblue", linewidth=2, markersize=8)
        ax.set_xlabel("Training Step", fontsize=12)
        ax.set_ylabel("PER (%) - 39 phones", fontsize=12)
        ax.set_title("Phoneme Error Rate vs Training Step", fontsize=13)
        ax.grid(True, alpha=0.3)
        ax.set_ylim(bottom=0)
        for s, p in zip(steps, pers):
            ax.annotate(f"{p:.1f}%", (s, p), textcoords="offset points",
                        xytext=(0, 10), ha="center", fontsize=9)
        plt.tight_layout()
        plt.show()
else:
    print("Checkpoint comparison skipped. Set SKIP_COMPARISON = False to run.")